# 회귀모형 독립감사: 식별변동과 군집추론

09·10번의 계산을 다른 수식으로 대조하고, 계산의 재현과 연구 설계의 타당성을 구분한다. 분석범위는 500m 기본모형·10개 업종·2개 결과다. 원 분석 코드를 실행하거나 함수를 가져오지 않는다. 원자료·기존 분석산출물은 수정하지 않으며 인터넷 요청도 하지 않는다.

감사 전에 정한 점검은 (1) 명시적 상권·분기 더미 OLS와 군집 공분산, (2) 잔차화된 설명변수의 군집별 집중도, (3) 업종마다 설명변수 부분 레버리지가 가장 큰 상권 한 개를 제외하는 동일 규칙, (4) 업종별 군집수로 추론했을 때의 변화, (5) 저장된 α에서 닫힌 형태 GLS로 GEE 추정식·공분산을 대조하는 것이다. 유의한 모형을 찾는 검색을 하지 않는다.

**구분:** 수치 불일치는 구현 오류 후보, 집중도·공간 상관·선택적 누락·인과 식별은 연구 한계다. 이 감사는 CR2/Satterthwaite나 공간 의존 추론의 검증을 대체하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os
for variable in ['OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'OMP_NUM_THREADS']:
    os.environ[variable] = '1'
import json, hashlib, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown
ROOT = PROJECT_ROOT
OUT = project_path('reports/20261005/audit_20261005/statistics')
OUT.mkdir(parents=True, exist_ok=True)
KEY = ['area_id', 'industry_id', 'quarter']
INPUT = project_path('dataset/analysis/food_industry_panel.csv')
source_paths = [INPUT, *[project_path('reports/20261005')/name for name in ['08_food_scope_metadata.json',
    '09_analysis_summary.json', '09_industry_effects.csv', '09_slope_covariances.csv',
    '09_omnibus_tests.csv', '09_model_sample_membership.csv', '09_activity_change_contrasts.csv',
    '10_gee_analysis_summary.json', '10_gee_diagnostics.csv', '10_gee_industry_effects.csv',
    '10_gee_sample_membership.csv']]]
hashes = {p.relative_to(ROOT).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths}
panel_hash = hashes[INPUT.relative_to(ROOT).as_posix()]
scope = project_json_loads((project_path('reports/20261005/08_food_scope_metadata.json')).read_text(encoding='utf-8'))
fe_manifest = project_json_loads((project_path('reports/20261005/09_analysis_summary.json')).read_text(encoding='utf-8'))
gee_manifest = project_json_loads((project_path('reports/20261005/10_gee_analysis_summary.json')).read_text(encoding='utf-8'))
assert scope['panel_sha256'] == fe_manifest['input_sha256'] == panel_hash
assert panel_hash in gee_manifest['input_sha256'].values()
for name, value in fe_manifest['output_sha256'].items():
    assert hashlib.sha256((project_path(name)).read_bytes()).hexdigest() == value, name
for name, value in gee_manifest['output_sha256'].items():
    assert hashlib.sha256((project_path('reports/20261005')/name).read_bytes()).hexdigest() == value, name
panel = pd.read_csv(INPUT, dtype={k:str for k in KEY})
assert not panel.duplicated(KEY).any()
labels = panel[['industry_id','industry']].drop_duplicates().set_index('industry_id')['industry'].to_dict()
industries = sorted(labels)
outcomes = ['log_sales','night_share_pct']
panel['log_activity'] = np.log1p(panel['known_performance_count_500m'])
panel['log_sales'] = np.log1p(panel['total_sales'].where(panel['total_sales'].ge(0)))
panel['night_share_pct'] = (100*panel['night_sales']/panel['total_sales']).where(
    panel['total_sales'].gt(0) & panel['night_sales'].ge(0) & panel['night_sales'].le(panel['total_sales']))
reference = pd.read_csv(project_path('reports/20261005/09_industry_effects.csv')).query("scenario == 'main_500'")
fe_keys = pd.read_csv(project_path('reports/20261005/09_model_sample_membership.csv'), dtype={k:str for k in KEY})
gee_keys = pd.read_csv(project_path('reports/20261005/10_gee_sample_membership.csv'), dtype={k:str for k in KEY})
pd.set_option('display.max_columns', 14)
pd.set_option('display.float_format', lambda v: f'{v:.6g}')
display(pd.DataFrame([{'panel_rows':len(panel),'input_sha256':panel_hash,
    'collection_run_id':scope['exposure_snapshot_linkage']['collection_run_id']}]))

## 1. 독립적으로 만든 표본과 직접 더미 회귀

공개 매출이 유효한 행에서 최소 2분기를 가진 상권×업종을 고른다. 각 업종에서 상권 더미 전체와 첫 분기를 뺀 분기 더미, log(1+공연 수)를 하나의 행렬에 넣어 최소제곱을 직접 푼다. 이것은 09번의 순차 평균 제거 경로를 재사용하지 않는 대조다. 두 결과의 표본이 같으면 하나의 설명행렬로 결과 두 열을 함께 계산한다.

In [ ]:
frames, sample_checks = {}, []
for outcome in outcomes:
    f = panel.loc[np.isfinite(panel[outcome]) & np.isfinite(panel['log_activity'])].copy()
    f = f.loc[f.groupby(['area_id','industry_id'])['quarter'].transform('nunique').ge(2)]
    f = f.sort_values(KEY).reset_index(drop=True)
    frames[outcome] = f
    actual = set(map(tuple, f[KEY].to_numpy()))
    saved = set(map(tuple, fe_keys.loc[(fe_keys.scenario=='main_500') & (fe_keys.outcome==outcome),KEY].to_numpy()))
    saved_gee = set(map(tuple, gee_keys.loc[gee_keys.outcome==outcome,KEY].to_numpy()))
    sample_checks.append(dict(outcome=outcome,nobs=len(f),exact_fe_keys=actual==saved,exact_gee_keys=actual==saved_gee))
assert all(row['exact_fe_keys'] and row['exact_gee_keys'] for row in sample_checks)
assert frames[outcomes[0]][KEY].equals(frames[outcomes[1]][KEY]), '공유 행렬 계산은 동일 표본에서만 허용'
frame = frames[outcomes[0]].copy()
frame[outcomes[1]] = frames[outcomes[1]][outcomes[1]].to_numpy()
display(pd.DataFrame(sample_checks))

In [ ]:
def direct_fit(part):
    z = np.column_stack([pd.get_dummies(part['area_id'],dtype=float).to_numpy(),
        pd.get_dummies(part['quarter'],drop_first=True,dtype=float).to_numpy()])
    x = part['log_activity'].to_numpy()
    y = part[outcomes].to_numpy()
    design = np.column_stack([z,x])
    coef, _, rank, _ = linalg.lstsq(design,y,cond=1e-10,lapack_driver='gelsd')
    resid = y-design@coef
    x_resid = x-z@linalg.lstsq(z,x,cond=1e-10,lapack_driver='gelsd')[0]
    # 직접 더미행렬의 기울기 투영 행: (X'X)^(-1)X'의 마지막 행
    projection = linalg.pinv(design,rtol=1e-10)[-1]
    influence = pd.DataFrame(projection[:,None]*resid, columns=outcomes).groupby(part['area_id'].to_numpy()).sum()
    return dict(part=part, beta=coef[-1], residual=resid, rank=int(rank),
        x_resid=x_resid, influence=influence, leverage=projection*x_resid)
blocks = {industry:direct_fit(frame.loc[frame.industry_id==industry].reset_index(drop=True)) for industry in industries}
area_ids = sorted(frame.area_id.unique())
G, N = len(area_ids), len(frame)
K = sum(b['rank'] for b in blocks.values())
correction = G/(G-1)*(N-1)/(N-K)
independent_covs, recomputed_rows, covariance_checks = {}, [], []
reference_cov = pd.read_csv(project_path('reports/20261005/09_slope_covariances.csv')).query("scenario == 'main_500'")
for j,outcome in enumerate(outcomes):
    influence = np.column_stack([blocks[i]['influence'][outcome].reindex(area_ids,fill_value=0) for i in industries])
    cov = correction * influence.T@influence
    independent_covs[outcome] = cov
    old_cov = reference_cov.loc[reference_cov.outcome==outcome].pivot(index='industry_id_a',columns='industry_id_b',values='covariance').loc[industries,industries].to_numpy()
    covariance_checks.append(dict(outcome=outcome,max_abs_covariance_difference=float(abs(cov-old_cov).max())))
    for k,industry in enumerate(industries):
        b = blocks[industry]
        old = reference.loc[(reference.outcome==outcome)&(reference.industry_id==industry)].iloc[0]
        beta, se = b['beta'][j], np.sqrt(cov[k,k])
        recomputed_rows.append(dict(outcome=outcome,industry_id=industry,industry=labels[industry],
            coefficient=beta,standard_error=se,p_value=2*stats.t.sf(abs(beta/se),G-1),
            coefficient_difference=beta-old.coefficient,standard_error_difference=se-old.standard_error))
recomputed = pd.DataFrame(recomputed_rows)
assert recomputed[['coefficient_difference','standard_error_difference']].abs().to_numpy().max()<1e-8
assert max(row['max_abs_covariance_difference'] for row in covariance_checks)<1e-8
display(recomputed[['outcome','industry','coefficient_difference','standard_error_difference']])
display(pd.DataFrame(covariance_checks))

**해석:** 일치는 현재 기본표본에서 FWL 계수와 상권 군집 교차업종 공분산이 명시적 더미 회귀와 같다는 뜻이다. 05번과의 기존 일치 여부를 정답 기준으로 사용하지 않았다. 이 수치 대조만으로 군집 독립성, 작은 유효 군집수에서의 추론, 누락의 무작위성 또는 인과성이 입증되지는 않는다.

## 2. 군집 수와 실제 식별변동의 집중도

부분 레버리지 비중은 상권별 잔차화한 공연변수 제곱합을 전체 제곱합으로 나눈 값이다. 1/Σ비중²는 정보가 얼마나 소수 상권에 모이는지 보여 주는 **집중도 기반 유효 군집수**다. 이는 Satterthwaite 자유도 추정치가 아니다. 공연변수가 전 기간 일정한 상권도 분기효과를 식별하는 비교자료로 기여할 수 있으므로 무조건 무정보로 해석하지 않는다.

In [ ]:
support_rows, cluster_rows = [], []
for industry,b in blocks.items():
    p = b['part']
    energy = pd.Series(b['x_resid']**2).groupby(p.area_id).sum()
    weights = (energy/energy.sum()).sort_values(ascending=False)
    ranges = p.groupby('area_id').known_performance_count_500m.agg(['min','max','nunique'])
    areas = p[['area_id','area_name','district']].drop_duplicates().set_index('area_id')
    support_rows.append(dict(industry_id=industry,industry=labels[industry],area_clusters=len(weights),
        raw_changing_areas=int(ranges['nunique'].gt(1).sum()),raw_constant_areas=int(ranges['nunique'].eq(1).sum()),
        top_area_id=weights.index[0],top_area_name=areas.loc[weights.index[0],'area_name'],
        top1_partial_leverage=float(weights.iloc[0]),top5_partial_leverage=float(weights.head(5).sum()),
        concentration_effective_clusters=float(1/(weights**2).sum()),
        areas_spanning_10_to_20=int((ranges['min'].le(10)&ranges['max'].ge(20)).sum()),
        residual_activity_sd=float(np.std(b['x_resid']))))
    for area,weight in weights.items():
        cluster_rows.append(dict(industry_id=industry,industry=labels[industry],area_id=area,
            area_name=areas.loc[area,'area_name'],district=areas.loc[area,'district'],partial_leverage=float(weight),
            score_influence_log_sales=float(b['influence'].loc[area,'log_sales']),
            score_influence_night_share=float(b['influence'].loc[area,'night_share_pct'])))
support = pd.DataFrame(support_rows)
cluster_detail = pd.DataFrame(cluster_rows)
display(support.drop(columns=['top_area_id','residual_activity_sd']))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,4.5))
positions = np.arange(len(support))
ax.bar(positions-.18,support.area_clusters,width=.36,label='Observed area clusters',color='#8BAABB')
ax.bar(positions+.18,support.concentration_effective_clusters,width=.36,label='Partial-leverage concentration effective clusters',color='#B65D43')
ax.set_xticks(positions,support.industry_id.str[-3:])
ax.set_xlabel('Food industry code suffix'); ax.set_ylabel('Cluster count / concentration diagnostic')
ax.legend(frameon=False,fontsize=8); ax.set_title('Nominal cluster count versus identification concentration')
fig.tight_layout(); fig.savefig(OUT/'identification_concentration.png',dpi=160)
plt.show(); plt.close(fig)

**해석:** 상권 군집수가 많더라도 공연변수의 잔차 변동이 몇 상권에 몰리면 정규·t 근사의 정확성을 그 숫자만으로 보장할 수 없다. 원 노트북의 20개 이상 군집·잔차 SD>0 기준은 이런 집중도를 검사하지 않는다. 아래에서는 p값에 의존하지 않고, 각 업종의 최상위 부분 레버리지 상권 하나를 제외하는 미리 정한 규칙만 비교한다.

In [ ]:
drop_rows=[]
for industry,b in blocks.items():
    top=support.loc[support.industry_id==industry].iloc[0]
    retained=b['part'].loc[b['part'].area_id!=top.top_area_id].reset_index(drop=True)
    reduced=direct_fit(retained)
    for j,outcome in enumerate(outcomes):
        old=reference.loc[(reference.outcome==outcome)&(reference.industry_id==industry)].iloc[0]
        drop_rows.append(dict(outcome=outcome,industry_id=industry,industry=labels[industry],
            removed_area_id=top.top_area_id,removed_area_name=top.top_area_name,
            removed_partial_leverage=top.top1_partial_leverage,original_coefficient=float(b['beta'][j]),
            omitted_coefficient=float(reduced['beta'][j]),coefficient_change=float(reduced['beta'][j]-b['beta'][j]),
            absolute_change_in_original_se=float(abs(reduced['beta'][j]-b['beta'][j])/old.standard_error),
            sign_flip=bool(np.sign(reduced['beta'][j])!=np.sign(b['beta'][j]))))
drop_top=pd.DataFrame(drop_rows)
display(drop_top[['outcome','industry','removed_area_name','original_coefficient','omitted_coefficient','absolute_change_in_original_se','sign_flip']])

**해석:** 상권 제외는 자료 오류를 발견했다는 뜻도, 더 나은 최종모형을 선택한 것도 아니다. 계수의 근거가 특정 상권에 집중되었는지 확인하는 영향도 진단이다. 원래 0 근처에 있는 계수의 부호 반전은 그 자체로 큰 불안정을 뜻하지 않으므로 변화량을 원 표준오차 단위로 함께 표시한다.

## 3. 업종별 군집수와 추론, 전체 검정 및 단위

원 09번은 전체 적층모형의 군집수 134와 설계 rank를 이용하는 CR1 보정과 자유도 133을 모든 계수에 쓴다. 이것은 실제 코드에 명시한 전역 보정의 구현이며 산술 오류로 분류하지 않는다. 그러나 개별 업종은 40~126개 상권만 포함한다. 업종을 단독 회귀했을 때의 CR1과 G업종−1 자유도는 현재 전역 방식과 다르며, 아래 대조는 작은 표본 근사에 대한 민감도를 보여 준다. 이것도 CR2나 정확한 추론은 아니다.

In [ ]:
local_rows=[]
for industry,b in blocks.items():
    g,n=len(b['influence']),len(b['part'])
    local_factor=g/(g-1)*(n-1)/(n-b['rank'])
    for j,outcome in enumerate(outcomes):
        beta=float(b['beta'][j]); se=float(np.sqrt(local_factor*np.sum(b['influence'][outcome]**2)))
        old=reference.loc[(reference.outcome==outcome)&(reference.industry_id==industry)].iloc[0]
        local_rows.append(dict(outcome=outcome,industry_id=industry,industry=labels[industry],
            reported_df=G-1,industry_df=g-1,reported_se=old.standard_error,industry_cr1_se=se,
            reported_p=old.p_value,industry_t_p=2*stats.t.sf(abs(beta/se),g-1),
            reported_bh=old.p_fdr_primary,se_ratio=se/old.standard_error))
local_inference=pd.DataFrame(local_rows)
local_inference['industry_bh']=multipletests(local_inference.industry_t_p,method='fdr_bh')[1]
display(local_inference[['outcome','industry','reported_df','industry_df','se_ratio','reported_p','industry_t_p','industry_bh']])

In [ ]:
omnibus_rows=[]
old_joint=pd.read_csv(project_path('reports/20261005/09_omnibus_tests.csv')).query("scenario=='main_500'")
for j,outcome in enumerate(outcomes):
    beta=np.array([blocks[i]['beta'][j] for i in industries])
    R=np.eye(10)[1:]-np.eye(10)[0]
    d=R@beta; V=R@independent_covs[outcome]@R.T
    F=float(d@linalg.solve(V,d,assume_a='sym')/9)
    old=old_joint.loc[old_joint.outcome==outcome].iloc[0]
    omnibus_rows.append(dict(outcome=outcome,f_statistic=F,p_value=stats.f.sf(F,9,G-1),
        rank_contrast_covariance=int(np.linalg.matrix_rank(V)),condition_number=float(np.linalg.cond(V)),
        f_difference=F-old.f_statistic))
omnibus=pd.DataFrame(omnibus_rows)
omnibus['holm']=multipletests(omnibus.p_value,method='holm')[1]
recomputed['bh']=multipletests(recomputed.p_value,method='fdr_bh')[1]
merged=recomputed.merge(reference[['outcome','industry_id','p_fdr_primary']],on=['outcome','industry_id'])
assert np.max(abs(merged.bh-merged.p_fdr_primary))<1e-8
changes=pd.read_csv(project_path('reports/20261005/09_activity_change_contrasts.csv'))
unit_checks=[]
for row in recomputed.itertuples():
    value=row.coefficient*np.log(21/11)
    if row.outcome=='log_sales': value=100*np.expm1(value)
    old=changes.loc[(changes.outcome==row.outcome)&(changes.industry_id==row.industry_id)].iloc[0]
    unit_checks.append(dict(outcome=row.outcome,industry_id=row.industry_id,absolute_difference=abs(value-old.estimate)))
assert max(row['absolute_difference'] for row in unit_checks)<1e-8
display(omnibus)
display(pd.DataFrame([{'main_fe_bh_below_005':int((recomputed.bh<.05).sum()),
    'local_cr1_bh_below_005':int((local_inference.industry_bh<.05).sum()),
    'largest_unit_conversion_difference':max(row['absolute_difference'] for row in unit_checks)}]))

**해석:** 기본 업종 기울기 20개 BH, 결과변수 2개 동일기울기 검정 Holm과 10→20개 비교 변환은 따로 확인했다. 원 분석은 유의/비유의의 차이를 곧 업종 간 차이로 해석하지 않으며, 변환을 산술평균 매출효과나 공연 1회당 효과로 표현하지 않는다. BH의 일반적인 보장은 검정통계의 의존구조에 조건이 있고, 서로 다른 모형군의 작은 p값을 골라 사용하면 별도 가족 보정의 의미가 깨진다. 현재 FE·GEE 모두 보정 후 유의항목이 없으며, 정확한 0이나 업종 동일성을 입증한 것은 아니다.

## 4. GEE의 독립 추정식 확인

Gaussian identity GEE는 α를 고정하면 일반화 최소제곱의 닫힌 해를 갖는다. 저장된 최종 α에서 각 군집의 R⁻¹=(I−αJ/[1+(m−1)α])/(1−α)를 직접 만들고, ΣX′R⁻¹X와 군집 score 외적으로 계수·robust 공분산을 재계산한다. 분산 척도는 계수와 robust sandwich에서 상쇄된다. 이 대조는 statsmodels GEE 추정기를 호출하지 않으며, α의 추정 자체를 독립 검증하는 것은 아니다.

In [ ]:
def gee_design(f):
    f=f.sort_values(KEY).reset_index(drop=True).copy()
    between=f.groupby(['area_id','industry_id']).log_activity.transform('mean')
    within=f.log_activity-between
    base=np.column_stack([np.ones(len(f)), f.district.eq('마포구').astype(float),
        pd.get_dummies(f.quarter,drop_first=True,dtype=float).to_numpy(), within,between])
    names=['intercept','district',*[f'q{q}' for q in sorted(f.quarter.unique())[1:]],'within','between']
    columns=[(i,name) for i in industries for name in names]
    matrix=np.column_stack([base*f.industry_id.eq(i).to_numpy()[:,None] for i in industries])
    return f,matrix,columns
gee_diag=pd.read_csv(project_path('reports/20261005/10_gee_diagnostics.csv')).set_index('outcome')
gee_reference=pd.read_csv(project_path('reports/20261005/10_gee_industry_effects.csv'))
gee_check_rows=[]
for outcome in outcomes:
    f,X,names=gee_design(frames[outcome]); y=f[outcome].to_numpy()
    alpha=float(gee_diag.loc[outcome,'alpha'])
    groups=[np.array(rows) for rows in f.groupby('area_id',sort=False).indices.values()]
    def inverse_correlation(a):
        return (a-alpha/(1+(len(a)-1)*alpha)*a.sum(axis=0))/(1-alpha)
    bread=np.zeros((X.shape[1],X.shape[1])); rhs=np.zeros(X.shape[1])
    for ix in groups:
        bread+=X[ix].T@inverse_correlation(X[ix]); rhs+=X[ix].T@inverse_correlation(y[ix])
    inv_bread=linalg.inv(bread); beta=linalg.solve(bread,rhs,assume_a='sym')
    residual=y-X@beta
    scores=np.array([X[ix].T@inverse_correlation(residual[ix]) for ix in groups])
    covariance=inv_bread@(scores.T@scores)@inv_bread.T
    for industry in industries:
        k=names.index((industry,'within'))
        old=gee_reference.loc[(gee_reference.outcome==outcome)&(gee_reference.industry_id==industry)].iloc[0]
        gee_check_rows.append(dict(outcome=outcome,industry_id=industry,industry=labels[industry],alpha=alpha,
            coefficient=float(beta[k]),standard_error=float(np.sqrt(covariance[k,k])),
            coefficient_difference=float(beta[k]-old.coefficient),
            standard_error_difference=float(np.sqrt(covariance[k,k])-old.standard_error),
            max_abs_estimating_equation=float(np.abs(scores.sum(axis=0)).max())))
gee_checks=pd.DataFrame(gee_check_rows)
assert gee_checks[['coefficient_difference','standard_error_difference']].abs().to_numpy().max()<1e-7
display(gee_checks[['outcome','industry','coefficient_difference','standard_error_difference','max_abs_estimating_equation']])

**해석:** 수치 일치는 저장된 α에서 GEE 계수와 sandwich를 재현했다는 뜻이다. 10번의 exchangeable 가정은 같은 상권의 서로 다른 업종·분기 쌍에 동일 α를 가정한다. 상권 간 공유 공연·동시 충격을 제거하지 않으며, 시간 내 평균을 뺀 변수라는 이유만으로 FE와 동등하거나 인과적이 되는 것도 아니다. 원 노트북이 이 차이를 설명하는 문구는 적절하다.

## 5. 문헌과 감사 범위

- [Cameron & Miller (2015), 저자 원문](https://facultysites.dss.ucdavis.edu/research/Cameron_Miller_JHR_2015_February.pdf): 군집 독립성, 작은 군집수와 t 근사의 한계.
- [Pustejovsky & Tipton (2018), 저자 설명 및 원논문](https://jepusto.com/publications/RVE-in-fixed-effects-models/index.html): 고정효과 아래 BRL/CR2 및 Satterthwaite 추론.
- [MacKinnon, Nielsen & Webb, 군집추론 실무 가이드](https://arxiv.org/abs/2205.03285): 부분 레버리지·영향도·군집 이질성 진단.

문헌은 감사 작성 시 별도로 열람했다. 이 노트북의 실행에는 네트워크가 필요 없다. CR2, wild cluster bootstrap, 공간 상관 강건 추론, 업종별 선택적 미제공 보정, GEE α 추정의 독립 재현은 이번 범위에서 수행하지 않았다. 확인하지 않은 방법이 통과했다거나 현재 결론을 뒤집을 것이라고 주장하지 않는다.

In [ ]:
exports={'sample_checks.csv':pd.DataFrame(sample_checks),'fe_independent_recalculation.csv':recomputed,
    'fe_covariance_checks.csv':pd.DataFrame(covariance_checks),'identification_support.csv':support,
    'cluster_partial_leverage.csv':cluster_detail,'omit_top_leverage_area.csv':drop_top,
    'industry_cluster_inference.csv':local_inference,'omnibus_recalculation.csv':omnibus,
    'effect_unit_checks.csv':pd.DataFrame(unit_checks),'gee_fixed_alpha_recalculation.csv':gee_checks}
for name,table in exports.items(): table.to_csv(OUT/name,index=False,encoding='utf-8-sig')
assert all(hashlib.sha256((project_path(name)).read_bytes()).hexdigest()==value for name,value in hashes.items())
summary=dict(input_sha256=hashes,python=platform.python_version(),packages={name:metadata.version(name)
    for name in ['numpy','pandas','scipy','statsmodels','nbformat']},
    direct_fe_max_coefficient_difference=float(recomputed.coefficient_difference.abs().max()),
    direct_fe_max_se_difference=float(recomputed.standard_error_difference.abs().max()),
    fixed_alpha_gee_max_coefficient_difference=float(gee_checks.coefficient_difference.abs().max()),
    fixed_alpha_gee_max_se_difference=float(gee_checks.standard_error_difference.abs().max()),
    fe_primary_bh_significant=int((recomputed.bh<.05).sum()),
    local_cr1_bh_significant=int((local_inference.industry_bh<.05).sum()),
    concentration_effective_clusters_min=float(support.concentration_effective_clusters.min()),
    concentration_effective_clusters_max=float(support.concentration_effective_clusters.max()),
    top1_partial_leverage_max=float(support.top1_partial_leverage.max()),
    top5_partial_leverage_max=float(support.top5_partial_leverage.max()),
    max_drop_change_original_se=float(drop_top.absolute_change_in_original_se.max()),
    scope='main_500 only; direct FE and fixed-alpha GEE; deterministic highest-leverage-area omission',
    unverified=['GEE alpha estimation','CR2/Satterthwaite','wild cluster bootstrap',
        'cross-area spatial dependence','selection from missing public sales','causal identification'])
(OUT/'audit_statistics_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([dict(check='FE direct-dummy coefficients and covariance',passed=True),
    dict(check='GEE closed-form coefficients and covariance at stored alpha',passed=True),
    dict(check='BH, Holm, effect-unit conversion',passed=True),dict(check='input files unchanged',passed=True)]))

**감사 결론:** 수치 구현과 추론 가정을 구분해서 읽는다. 독립 재계산의 일치는 확인된 구현 근거다. 식별변동 집중과 군집 독립성의 제한은 별도로 남는다. 상권 한 개 제외 결과와 업종별 군집수 추론 결과는 최종모형을 교체하기 위한 선택 기준이 아니라, 현재 추정의 지원범위와 불확실성을 보고하기 위한 진단이다.